# Previsão de jogos da Liga Portugal: modelos

**Fase 3: referências.** Antes de treinar qualquer modelo, medem-se as regras simples
que ele tem de bater, na época de validação (2024/25).

Medidas (quanto mais baixo melhor, exceto o acerto):
- **Acerto:** o resultado mais provável aconteceu?
- **Log loss:** média de -log(probabilidade dada ao resultado real). Adivinhar ao acaso dá 1,10.
- **Brier:** soma dos quadrados entre as probabilidades e o que aconteceu (1 no real, 0 nos outros).

In [ ]:
import sqlite3
from pathlib import Path
from contextlib import closing

import numpy as np
import pandas as pd

PASTA_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
with closing(sqlite3.connect(PASTA_PROJETO / "dados" / "futebol.db")) as ligacao:
    variaveis = pd.read_sql("SELECT * FROM variaveis", ligacao, parse_dates=["Data"])

treino = variaveis[variaveis["Uso"] == "Treino"]
validacao = variaveis[variaveis["Uso"] == "Validação"]
print(f"Treino: {len(treino)} jogos | Validacao: {len(validacao)} jogos")

Treino: 1836 jogos | Validacao: 306 jogos


In [14]:
RESULTADOS = ["H", "D", "A"]

def avaliar(nome,probabilidades,resultado_real):
    probabilidades = np.asarray(probabilidades, dtype=float)
    real = pd.Series(resultado_real).to_numpy()

    previsto = np.array(RESULTADOS)[probabilidades.argmax(axis=1)]
    acerto = (previsto == real).mean()

    real_01 = np.column_stack([(real == r).astype(float) for r in RESULTADOS])
    brier = ((probabilidades - real_01) ** 2).sum(axis=1).mean()
    
    prob_do_real = (probabilidades * real_01).sum(axis=1)
    perda = -np.log(np.clip(prob_do_real, 1e-15, 1)).mean()

    return {"Modelo": nome, "Acerto": round(acerto, 3), "LogLoss": round(perda, 3), "Brier": round(brier, 3)}


In [15]:
frequencias = treino["Resultado"].value_counts(normalize=True).reindex(RESULTADOS)
print(frequencias.round(3))
prob_frequencias = np.tile(frequencias.to_numpy(), (len(validacao), 1))
r1 = avaliar("Frequências do treino", prob_frequencias, validacao["Resultado"])

Resultado
H    0.436
D    0.237
A    0.327
Name: proportion, dtype: float64


In [16]:
# Os limites dos escalões vêm só do treino
_, limites = pd.qcut(treino["DifElo"], 5, retbins=True)
limites[0], limites[-1] = -np.inf, np.inf
escalao_treino = pd.cut(treino["DifElo"], limites)
tabela_elo = pd.crosstab(escalao_treino, treino["Resultado"], normalize="index")[RESULTADOS]
display(tabela_elo.round(2))

escalao_validacao = pd.cut(validacao["DifElo"], limites)
prob_elo = tabela_elo.loc[escalao_validacao].to_numpy()
r2 = avaliar("Elo por escalões", prob_elo, validacao["Resultado"])

Resultado,H,D,A
DifElo,,,
"(-inf, -99.108]",0.15,0.17,0.68
"(-99.108, -23.172]",0.37,0.29,0.34
"(-23.172, 22.415]",0.43,0.30,0.27
"(22.415, 99.085]",0.44,0.29,0.26
"(99.085, inf]",0.79,0.13,0.08


In [17]:
prob_odds = validacao[["ProbOddsCasa", "ProbOddsEmpate", "ProbOddsFora"]].to_numpy()
r3 = avaliar("Odds (Bet365)", prob_odds, validacao["Resultado"])

In [18]:
referencias = pd.DataFrame([r1, r2, r3])
referencias

,Modelo,Acerto,LogLoss,Brier
0,Frequências do treino,0.431,1.077,0.652
1,Elo por escalões,0.516,0.975,0.580
2,Odds (Bet365),0.556,0.935,0.553


In [22]:
import sys
sys.path.insert(0, str(PASTA_PROJETO / "etl"))
from func.variaveis import colunas_do_modelo

colunas = colunas_do_modelo(variaveis)
X_treino, y_treino = treino[colunas], treino["Resultado"]
X_validacao, y_validacao = validacao[colunas], validacao["Resultado"]
print(f"{len(colunas)} variaveis | treino: {len(X_treino.shape)} | validacao: {len(X_validacao.shape)}")

24 variaveis | treino: 2 | validacao: 2


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(X_treino, y_treino)

print("Treinado. Classes:", modelo.classes_)

Treinado. Classes: ['A' 'D' 'H']


In [26]:
prob_modelo = pd.DataFrame(modelo.predict_proba(X_validacao), columns=modelo.classes_)
prob_modelo = prob_modelo[RESULTADOS]
prob_modelo.head()

,H,D,A
0,0.860942,0.100234,0.038825
1,0.383750,0.351540,0.264710
2,0.432269,0.272558,0.295174
3,0.802768,0.116515,0.080718
4,0.372906,0.377715,0.249379


In [27]:
r4 = avaliar("Modelo de regressão logística", prob_modelo, y_validacao)
pd.DataFrame([r1, r2, r4, r3])

,Modelo,Acerto,LogLoss,Brier
0,Frequências do treino,0.431,1.077,0.652
1,Elo por escalões,0.516,0.975,0.580
2,Modelo de regressão logística,0.533,0.948,0.562
3,Odds (Bet365),0.556,0.935,0.553


In [28]:
pesos = pd.DataFrame(modelo[-1].coef_.T, index=colunas, columns=modelo.classes_)[RESULTADOS]
pesos.round(2).sort_values(by="H", ascending=False)

,H,D,A
DifElo,0.33,-0.07,-0.26
EloCasa,0.28,-0.11,-0.17
ForaPontosUlt5,0.09,0.02,-0.11
EquilibrioElo,0.09,-0.17,0.07
CasaRematesBalizaUlt5,0.07,-0.09,0.02
ForaGolosSofridosUlt5,0.05,0.01,-0.06
CasaPromovida,0.04,0.01,-0.05
ForaRematesBalizaSofridosUlt5,0.04,-0.01,-0.03
DifRematesBalizaUlt5,0.04,0.01,-0.05
DifGolosMarcadosUlt5,0.03,-0.01,-0.02
